# 02. Validação e Inspeção do Dataset para Fine-Tuning (Gemini)

Este notebook carrega os pares de diálogo gerados em `data/processed/train_gemini.jsonl` e `data/processed/val_gemini.jsonl` e fornece métricas de qualidade, distribuição de tamanho e amostras interativas das conversas.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from collections import Counter

data_dir = Path('../data/processed')
train_file = data_dir / 'train_gemini.jsonl'
val_file = data_dir / 'val_gemini.jsonl'

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
    return records

train_data = load_jsonl(train_file)
val_data = load_jsonl(val_file)

print(f'Exemplos de Treino: {len(train_data)}')
print(f'Exemplos de Validação: {len(val_data)}')
print(f'Total Combinado: {len(train_data) + len(val_data)}')

## 1. Estatísticas de Tamanho e Distribuição
Avaliando o comprimento médio das mensagens de contexto (`user`) e das suas respostas (`model`).

In [ ]:
stats = []
for split_name, dataset in [('Treino', train_data), ('Validação', val_data)]:
    for item in dataset:
        user_content = item['messages'][0]['content']
        model_content = item['messages'][1]['content']
        stats.append({
            'split': split_name,
            'user_len_chars': len(user_content),
            'user_word_count': len(user_content.split()),
            'model_len_chars': len(model_content),
            'model_word_count': len(model_content.split()),
        })

df_stats = pd.DataFrame(stats)
df_stats.groupby('split')[['user_word_count', 'model_word_count', 'model_len_chars']].describe().T

## 2. Inspeção de Exemplos Reais de Conversa
Veja abaixo como o contexto do grupo foi montado e qual foi a resposta real do Yan (`model`):

In [ ]:
for i, item in enumerate(train_data[:5]):
    user_msg = item['messages'][0]['content']
    model_msg = item['messages'][1]['content']
    print(f'=== EXEMPLO {i+1} ===')
    print(f'>>> CONTEXTO DO GRUPO (user):\n{user_msg}\n')
    print(f'>>> RESPOSTA DO YAN (model):\n{model_msg}')
    print('-' * 60 + '\n')

## 3. Expressões e Vocabulário mais Frequentes do Yan
Identificando termos típicos que definem o tom de voz da persona.

In [ ]:
words = []
for item in train_data + val_data:
    model_text = item['messages'][1]['content'].lower()
    # Remove pontuação simples para contagem
    for w in model_text.replace('\n', ' ').split():
        cleaned_w = ''.join(c for c in w if c.isalnum())
        if len(cleaned_w) > 2:
            words.append(cleaned_w)

counter = Counter(words)
pd.DataFrame(counter.most_common(25), columns=['Palavra', 'Frequência'])